In [ ]:
 !pip install openai

In [14]:
import json
import os
import openai
import requests

StatementMeta(entdatamtc, 3, 14, Finished, Available)

In [43]:


# Setting up the deployment name
deployment_name = 'CX-text-davinci-003'
#deployment_name = 'cx-gpt4test'

# This is set to `azure`
openai.api_type = "azure"

# The API key for your Azure OpenAI resource.
openai.api_key = '0038017ad6384d02a7c77a15b3d3c485'

# The base URL for your Azure OpenAI resource. e.g. "https://<your resource name>.openai.azure.com"
openai.api_base = 'https://gen-entdata-openai-np.openai.azure.com'

# Currently OPENAI API have the following versions available: 2022-12-01
openai.api_version = '2023-03-15-preview'



StatementMeta(entdatamtc, 3, 43, Finished, Available)

In [50]:
# Give your prompt here
#prompt = "Hello world"

prompt = """
### Microsoft SQL tables, with their properties:
#
# oracle.XXNPS_SALES_ORDER_DLAKE_MTC ([H_OPCO]
,[H_CUSTOMER_NAME]
,[H_SHIP_TO_COUNTRY]
,[L_LINE_TOTAL_IN_FUNCTIONAL_CURRENCY]
,[L_WAREHOUSE]
,[INVOICE_AMOUNT]
,[INVOICE_DATE]
,[ORDER_STATUS]
,[L_ITEM_DESCRIPTION]
,[L_ITEM_NUMBER])
### A query to list the total revenue by month and year

# Do not end with ;
SELECT 
"""

try:
    # Create a completion for the provided prompt and parameters
    # To know more about the parameters, checkout this documentation: https://learn.microsoft.com/en-us/azure/cognitive-services/openai/reference
    completion = openai.Completion.create(
                    prompt=prompt,
                    temperature=0,
                    max_tokens=300,
                    engine=deployment_name)

    # print the completion
    query='SELECT '+ completion.choices[0].text.strip(" \n")
    print(query)
    #print(completion)
    
    # Here indicating if the response is filtered
    if completion.choices[0].finish_reason == "content_filter":
        print("The generated content is filtered.")
        
except openai.error.APIError as e:
    # Handle API error here, e.g. retry or log
    print(f"OpenAI API returned an API Error: {e}")

except openai.error.AuthenticationError as e:
    # Handle Authentication error here, e.g. invalid API key
    print(f"OpenAI API returned an Authentication Error: {e}")

except openai.error.APIConnectionError as e:
    # Handle connection error here
    print(f"Failed to connect to OpenAI API: {e}")

except openai.error.InvalidRequestError as e:
    # Handle connection error here
    print(f"Invalid Request Error: {e}")

except openai.error.RateLimitError as e:
    # Handle rate limit error
    print(f"OpenAI API request exceeded rate limit: {e}")

except openai.error.ServiceUnavailableError as e:
    # Handle Service Unavailable error
    print(f"Service Unavailable: {e}")

except openai.error.Timeout as e:
    # Handle request timeout
    print(f"Request timed out: {e}")

StatementMeta(entdatamtc, 3, 50, Finished, Available)

SELECT YEAR(INVOICE_DATE) AS [Year], 
    MONTH(INVOICE_DATE) AS [Month], 
    SUM(L_LINE_TOTAL_IN_FUNCTIONAL_CURRENCY) AS [Total Revenue]
FROM oracle.XXNPS_SALES_ORDER_DLAKE_MTC
GROUP BY YEAR(INVOICE_DATE), MONTH(INVOICE_DATE)
ORDER BY YEAR(INVOICE_DATE), MONTH(INVOICE_DATE)


In [45]:
# Add required imports
import com.microsoft.spark.sqlanalytics
from com.microsoft.spark.sqlanalytics.Constants import Constants
from pyspark.sql.functions import col
#query= 'SELECT H_SHIP_TO_COUNTRY,SUM(INVOICE_AMOUNT) AS Total_Invoice_Amount FROM oracle.XXNPS_SALES_ORDER_DLAKE_MTC GROUP BY H_SHIP_TO_COUNTRY'

dfToReadFromQueryAsArgument = (spark.read
                     # Name of the SQL Dedicated Pool or database where to run the query
                     # Database can be specified as a Spark Config - spark.sqlanalyticsconnector.dw.database or as a Constant - Constants.DATABASE
                      .option(Constants.DATABASE, "Curated")
                     # If `Constants.SERVER` is not provided, the `Curated` from the three-part table name argument
                     # to `synapsesql` method is used to infer the Synapse Dedicated SQL End Point.
                     .option(Constants.SERVER, "entdatasynapseprod.sql.azuresynapse.net")
                     # Defaults to storage path defined in the runtime configurations
                    #  .option(Constants.TEMP_FOLDER, "abfss://test@entdatasynapseprod.dfs.core.windows.net/MTCSession_SynapseSQL")
                     # query from which data will be read
                     .synapsesql(query)
)

# Show contents of the dataframe

display(dfToReadFromQueryAsArgument)

StatementMeta(entdatamtc, 3, 45, Finished, Available)

SynapseWidget(Synapse.DataFrame, 3c6c6bc5-d68f-4433-8570-0a0527401b1d)